# Chat libre -- pruebas de los modelos LLM

Prueba los modelos de Ollama que usa `lora_brain` **sin ROS 2**: `llama_client.py`,
`memoria_episodica.py`, `personalidad.py` y `registro_chat.py` son librerias Python
normales (ver la nota de `llama_client.py` sobre "que es nodo vs que es libreria"),
asi que se importan tal cual y lo que se mide aqui es exactamente lo que corre en
el robot.

| Variable | Default | Rol |
|---|---|---|
| `CHAT_MODEL` | `lora-chat-libre-v4` | respuestas del Chat libre |
| `TRIVIA_MODEL` | `lora-trivia` | reacciones del juego de emociones |
| `SALIDA_TRIVIA_MODEL` | `lora-salida-trivia-v2` | RESPUESTA vs SALIR a mitad de trivia |

**Requisitos:** Ollama corriendo (`http://localhost:11434`) con los 3 modelos
importados (`ollama create`, no `ollama pull`) y `pip install requests rank_bm25 psutil`.
Conviene `OLLAMA_KEEP_ALIVE=-1` para que no se descarguen de memoria entre celdas.

El foco es el **modulo de Chat libre**: `OrchestratorNode.responder()`, que es
memoria episodica BM25 + `CHAT_MODEL`, **sin RAG y sin system prompt** (el
comportamiento sale del fine-tuning).

## 1. Entorno

Se mete `src/lora_brain` en el `sys.path` para importar el paquete sin `colcon build`.
Las variables de entorno se fijan **antes** de importar `llama_client`, porque ahi
los nombres de modelo se leen a nivel de modulo.

In [ ]:
import os
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "Jupyter-notebook" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src" / "lora_brain"))

# Mismos defaults que llama_client.py -- cambiar aqui para probar otro fine-tune.
os.environ.setdefault("CHAT_SERVER_HOST", "http://localhost:11434")
os.environ.setdefault("CHAT_MODEL", "lora-chat-libre-v4")
os.environ.setdefault("TRIVIA_MODEL", "lora-trivia")
os.environ.setdefault("SALIDA_TRIVIA_MODEL", "lora-salida-trivia-v2")

# El notebook no debe ensuciar el registro real del robot (~/.lora/...), que es
# lo que lee memoria_episodica.py. Se usa una copia aparte.
REGISTRO_NB = Path.home() / ".lora" / "chat_libre_training" / "notebook.jsonl"
os.environ["CHAT_LIBRE_REGISTRO"] = str(REGISTRO_NB)

from lora_brain import llama_client, memoria_episodica, registro_chat  # noqa: E402
from lora_brain.personalidad import construir_personalidad, obtener_system_prompt  # noqa: E402

print("host               :", llama_client.CHAT_SERVER_HOST)
print("CHAT_MODEL         :", llama_client.CHAT_MODEL)
print("TRIVIA_MODEL       :", llama_client.TRIVIA_MODEL)
print("SALIDA_TRIVIA_MODEL:", llama_client.SALIDA_TRIVIA_MODEL)
print("registro           :", registro_chat.REGISTRO)

## 2. Ollama esta arriba y tiene los modelos

Si falta alguno, la celda lo dice en vez de dejar que el error aparezca recien
al generar.

In [ ]:
import requests

ESPERADOS = [
    llama_client.CHAT_MODEL,
    llama_client.TRIVIA_MODEL,
    llama_client.SALIDA_TRIVIA_MODEL,
]

try:
    tags = requests.get(f"{llama_client.CHAT_SERVER_HOST}/api/tags", timeout=10).json()
    instalados = [m["name"] for m in tags.get("models", [])]
except requests.RequestException as e:
    instalados = []
    print(f"Ollama no responde en {llama_client.CHAT_SERVER_HOST}: {e}")
    print("Arrancarlo con `ollama serve` (o `OLLAMA_KEEP_ALIVE=-1 ollama serve`).")

for m in instalados:
    print(" ", m)

# Ollama lista 'nombre:latest'; los modelos se invocan sin el tag.
base = {n.split(":")[0] for n in instalados}
for esperado in ESPERADOS:
    estado = "OK" if esperado.split(":")[0] in base else "FALTA (ollama create)"
    print(f"{esperado:<28} {estado}")

## 3. Warmup

Igual que `_precargar_modelos()` del orquestador: la primera llamada paga la carga
desde disco (15-28 s en el peor caso en la Pi), y si no se precalienta esa demora
contamina la primera medicion de latencia.

In [ ]:
import time

for modelo in ESPERADOS[:2]:  # el de salida_trivia se precalienta en su propia seccion
    t0 = time.perf_counter()
    try:
        llama_client.generar_respuesta(
            [{"role": "user", "content": "hola"}], max_tokens=1, modelo=modelo)
        print(f"{modelo:<28} {time.perf_counter() - t0:5.1f}s")
    except Exception as e:
        print(f"{modelo:<28} fallo: {e}")

## 4. Chat libre: el prompt real

`responder()` del orquestador, replicado aqui sin ROS 2 ni `perf_monitor`.
Un solo mensaje `user`, prefijado por el recuerdo BM25 cuando hay uno:

```python
recuerdo = memoria_episodica.buscar_relevante(mensaje_usuario)
contenido = f"{recuerdo}\n{mensaje_usuario}" if recuerdo else mensaje_usuario
mensajes  = [{"role": "user", "content": contenido}]
```

`temperature=0.3` y `max_tokens=50` son los defaults de `generar_respuesta()`,
los que usa el robot.

In [ ]:
def responder(mensaje_usuario, modelo=None, temperature=0.3, max_tokens=50,
              usar_memoria=True, registrar=False, mostrar_prompt=False):
    """Equivalente a OrchestratorNode.responder(). Devuelve (texto, segundos).

    `registrar=False` por defecto: a diferencia del robot, aqui no se quiere
    que cada prueba entre a la memoria episodica y cambie el prompt de la
    prueba siguiente.
    """
    modelo = modelo or llama_client.CHAT_MODEL
    recuerdo = memoria_episodica.buscar_relevante(mensaje_usuario) if usar_memoria else None
    contenido = f"{recuerdo}\n{mensaje_usuario}" if recuerdo else mensaje_usuario
    mensajes = [{"role": "user", "content": contenido}]

    if mostrar_prompt:
        print("--- prompt ---")
        print(contenido)
        print("--------------")

    t0 = time.perf_counter()
    texto = llama_client.generar_respuesta(
        mensajes, temperature=temperature, max_tokens=max_tokens, modelo=modelo).strip()
    dt = time.perf_counter() - t0

    if registrar:
        registro_chat.registrar(mensaje_usuario, texto, modelo)
    return texto, dt


texto, dt = responder("hola, como te llamas?", mostrar_prompt=True)
print(f"[{dt:.1f}s] {texto}")

### 4.1 Tanda de prueba

Las respuestas del Chat libre deberian ser cortas (el fine-tune apunta a ~25
palabras), en espanol natural, sin "como IA/asistente" ni cerrar preguntando.
La columna de palabras es para ver de un golpe si el modelo se va de largo.

In [ ]:
PRUEBAS = [
    "hola, como estas?",
    "me llamo Adrian, encantado",
    "estoy medio cansado hoy",
    "que te gusta hacer?",
    "cual es la capital de Peru?",
    "me pelee con un amigo y no se que hacer",
    "contame un chiste",
    "gracias, chau",
]

resultados = []
for mensaje in PRUEBAS:
    try:
        texto, dt = responder(mensaje)
    except Exception as e:
        texto, dt = f"<error: {e}>", float("nan")
    resultados.append({
        "usuario": mensaje,
        "respuesta": texto,
        "palabras": len(texto.split()),
        "seg": round(dt, 2),
    })
    print(f"> {mensaje}\n  [{dt:4.1f}s | {len(texto.split()):2d} pal] {texto}\n")

In [ ]:
# Tabla, si hay pandas. Si no, el print de arriba ya alcanza.
try:
    import pandas as pd

    df = pd.DataFrame(resultados)
    display(df)
    print(f"mediana: {df['seg'].median():.1f}s   max: {df['seg'].max():.1f}s"
          f"   palabras (media): {df['palabras'].mean():.1f}")
except ImportError:
    print("sin pandas")

## 5. Comparar modelos

El punto de esta celda es decidir entre fine-tunes (`lora-chat-libre-v4` vs una
v5, vs el `qwen2.5:0.5b` base) con el **mismo prompt** para cada uno. Editar
`CANDIDATOS` con lo que este instalado; los que falten se saltean solos.

Referencia del README: el modelo elegido debe quedar en **1-3 s** por respuesta
y ~484 MB de RAM. Ojo que en esta maquina los tiempos seran mejores que en la Pi.

In [ ]:
CANDIDATOS = [
    llama_client.CHAT_MODEL,
    # "lora-chat-libre-v5",
    # "qwen2.5:0.5b",
]
CANDIDATOS = [m for m in CANDIDATOS if m.split(":")[0] in base]

comparacion = []
for modelo in CANDIDATOS:
    print(f"\n=== {modelo} ===")
    try:  # warmup aparte, para no cargarle el disco a la primera medicion
        llama_client.generar_respuesta(
            [{"role": "user", "content": "hola"}], max_tokens=1, modelo=modelo)
    except Exception as e:
        print(f"  no se pudo precargar: {e}")
        continue

    for mensaje in PRUEBAS[:5]:
        try:
            texto, dt = responder(mensaje, modelo=modelo, usar_memoria=False)
        except Exception as e:
            texto, dt = f"<error: {e}>", float("nan")
        comparacion.append({
            "modelo": modelo,
            "usuario": mensaje,
            "respuesta": texto,
            "palabras": len(texto.split()),
            "seg": round(dt, 2),
        })
        print(f"  > {mensaje}\n    [{dt:4.1f}s] {texto}")

In [ ]:
try:
    import pandas as pd

    dfc = pd.DataFrame(comparacion)
    if not dfc.empty:
        display(dfc.groupby("modelo")[["seg", "palabras"]].agg(["mean", "median", "max"]))
        display(dfc.pivot(index="usuario", columns="modelo", values="respuesta"))
except ImportError:
    print("sin pandas")

## 6. Memoria episodica (BM25)

El unico contexto que recibe el Chat libre. `buscar_relevante()` exige **>= 2
palabras de contenido en comun** (`_OVERLAP_MINIMO`), no "score > 0": con el
umbral viejo enganchaba temas sin relacion.

Dos cosas que conviene tener a la vista al juzgar las respuestas del modelo:

- **No hay stemming.** `_tokenizar()` solo baja a minuscula, saca stopwords y
  descarta tokens de <= 2 caracteres, asi que `partido` y `partidos` son palabras
  distintas y no suman overlap. Varias consultas que "deberian" enganchar no lo
  hacen, y es por esto, no por BM25.
- **El recuerdo es la respuesta vieja de Lora**, no el mensaje del usuario, y
  entra recortada a 140 caracteres con el prefijo `(Ya hablamos de esto antes: ...)`.

Se escriben episodios de prueba en el registro del notebook (no en el del robot)
para ver cuando engancha y cuando no.

In [ ]:
EPISODIOS_DEMO = [
    ("me gusta mucho el futbol", "El futbol es lindo, yo miraria todos los partidos."),
    ("tengo un perro que se llama Rocco", "Rocco suena a un perro con caracter."),
    ("trabajo programando en python", "Programar en Python esta bueno, es prolijo."),
]

REGISTRO_NB.parent.mkdir(parents=True, exist_ok=True)
REGISTRO_NB.write_text("", encoding="utf-8")  # arrancar limpio
for usuario, respuesta in EPISODIOS_DEMO:
    registro_chat.registrar(usuario, respuesta, llama_client.CHAT_MODEL)

memoria_episodica._cache["mtime"] = None  # el cache es por mtime; forzar reindexado

CONSULTAS = [
    "me gusta el futbol, viste?",         # engancha: gusta + futbol
    "como esta Rocco, tu perro?",         # engancha: rocco + perro
    "viste el partido de futbol ayer?",   # NO engancha: "partido" != "partidos"
    "programas en python todavia?",       # NO engancha: "programas" != "programando"
    "que opinas del clima?",              # NO engancha: tema sin relacion
    "python",                             # NO engancha: 1 token, overlap 1 < 2
]

# Se imprime el overlap, que es lo que decide. El tokenizador no hace stemming
# (ver _tokenizar), asi que singular/plural y las distintas formas de un verbo
# cuentan como palabras diferentes: de ahi los "NO engancha" de arriba, que son
# el precio de BM25 sin embeddings.
_, episodios, corpus_tok = memoria_episodica._obtener_indice()
for consulta in CONSULTAS:
    query_tok = set(memoria_episodica._tokenizar(consulta))
    mejor = max((len(query_tok & set(t)) for t in corpus_tok), default=0)
    print(f"> {consulta}")
    print(f"  tokens  = {sorted(query_tok)}")
    print(f"  overlap = {mejor} (minimo {memoria_episodica._OVERLAP_MINIMO})")
    print(f"  {memoria_episodica.buscar_relevante(consulta)}\n")

In [ ]:
# Con memoria vs sin memoria, mismo mensaje: se ve cuanto la usa el modelo.
# Tiene que ser un mensaje que SI enganche (overlap >= 2 en la celda anterior),
# si no las dos corridas mandan el mismo prompt y la comparacion no dice nada.
mensaje = "me gusta el futbol, viste?"
for usar in (False, True):
    texto, dt = responder(mensaje, usar_memoria=usar)
    print(f"memoria={usar!s:<5} [{dt:4.1f}s] {texto}")

## 7. Conversacion interactiva

Un turno por celda, escribiendo en `MENSAJE`. `registrar=True` para que el turno
entre a la memoria episodica y el siguiente lo pueda usar -- es lo que hace el
robot en cada turno de Chat libre.

Vale recordar que el Chat libre **no manda historial**: cada llamada es un solo
mensaje `user`, y toda la continuidad pasa por BM25.

In [ ]:
MENSAJE = "me gusta jugar al futbol los domingos"

texto, dt = responder(MENSAJE, registrar=True, mostrar_prompt=True)
print(f"[{dt:.1f}s] {texto}")

### 7.1 Streaming token por token

`generar_respuesta()` soporta `on_token`. El orquestador no lo expone por ROS 2
(`Speak.srv` es request/response completo), pero aqui sirve para ver si el modelo
se traba a mitad de la generacion o si corta por `max_tokens`.

In [ ]:
t0 = time.perf_counter()
primer_token = [None]


def mostrar(delta):
    if primer_token[0] is None:
        primer_token[0] = time.perf_counter() - t0
    print(delta, end="", flush=True)


texto = llama_client.generar_respuesta(
    [{"role": "user", "content": "contame algo de ti en pocas palabras"}],
    on_token=mostrar)
print(f"\n\nprimer token: {primer_token[0]:.2f}s   total: {time.perf_counter() - t0:.2f}s")

## 8. Los otros dos modelos

No son Chat libre, pero conviene verificarlos en la misma corrida porque comparten
RAM y CPU con `CHAT_MODEL`.

### 8.1 `SALIDA_TRIVIA_MODEL` -- clasificador RESPUESTA / SALIR

`clasificar_salida_trivia()` devuelve `True` (SALIR), `False` (RESPUESTA) o `None`
si no se pudo decidir; con `None` el orquestador cae a listas de palabras clave.
Un `None` en esta tabla no es un crash, es el modelo contestando cualquier cosa.

In [ ]:
CASOS_SALIDA = [
    ("Cual es la capital de Francia?", "Paris", False),
    ("Cual es la capital de Francia?", "Madrid", False),            # mal, pero responde
    ("Cual es la capital de Francia?", "no quiero jugar mas", True),
    ("Cual es la capital de Francia?", "me pelee con mi hermano", True),
    ("Cual es la capital de Francia?", "ni idea", False),
    ("Que opinas de la inteligencia artificial?", "me parece util para estudiar", False),
    ("Cual es la capital de Francia?", "hablemos de otra cosa", True),
]

aciertos = 0
for pregunta, mensaje, esperado in CASOS_SALIDA:
    t0 = time.perf_counter()
    obtenido = llama_client.clasificar_salida_trivia(pregunta, mensaje)
    dt = time.perf_counter() - t0
    ok = obtenido == esperado
    aciertos += ok
    etiqueta = {True: "SALIR", False: "RESPUESTA", None: "None (fallback keywords)"}
    print(f"{'OK ' if ok else 'MAL'} [{dt:4.1f}s] {mensaje!r}\n"
          f"      esperado={etiqueta[esperado]:<10} obtenido={etiqueta[obtenido]}")
print(f"\n{aciertos}/{len(CASOS_SALIDA)} correctos")

### 8.2 `TRIVIA_MODEL` -- reacciones del juego de emociones

A diferencia del Chat libre, aca si hay system prompt: `_mensajes_con_personalidad()`
lo pide a `obtener_system_prompt()`, que lo **omite** si el modelo ya trae la
personalidad horneada por fine-tuning (`lora-trivia` esta en esa lista, asi que el
prompt sale vacio). `max_tokens=20`, porque la reaccion tiene que ser de ~5 palabras.

In [ ]:
persona_str = construir_personalidad()
print("personalidad:", persona_str)
sp_trivia = obtener_system_prompt(persona_str, modelo=llama_client.TRIVIA_MODEL)
sp_chat = obtener_system_prompt(persona_str, modelo=llama_client.CHAT_MODEL)
print(f"system prompt para {llama_client.TRIVIA_MODEL}: "
      f"{'(vacio: personalidad horneada)' if not sp_trivia else repr(sp_trivia[:80])}")
print(f"system prompt para {llama_client.CHAT_MODEL}: "
      f"{'(vacio)' if not sp_chat else repr(sp_chat[:80]) + '...'}")


def reaccion_emocion(objetivo, detectada, acerto):
    """Equivalente a OrchestratorNode.comentar_resultado_emocion()."""
    if acerto:
        instruccion = (
            f"Le pediste al usuario que pusiera cara de {objetivo.lower()} y la camara "
            f"detecto justo esa cara. Celebraselo en maximo 5 palabras.")
    else:
        instruccion = (
            f"Le pediste al usuario que pusiera cara de {objetivo.lower()} pero la camara "
            f"detecto cara de {detectada.lower()}. Dile en maximo 5 palabras que no era esa, "
            "con humor.")
    contenido = f"{instruccion} No vuelvas a pedir otra cara ni hagas otra pregunta."
    mensajes = []
    if sp_trivia:
        mensajes.append({"role": "system", "content": sp_trivia})
    mensajes.append({"role": "user", "content": contenido})
    return llama_client.generar_respuesta(
        mensajes, modelo=llama_client.TRIVIA_MODEL, max_tokens=20).strip()


print()
for objetivo, detectada, acerto in [
    ("Feliz", "Feliz", True),
    ("Triste", "Feliz", False),
    ("Sorpresa", "Enojo", False),
]:
    t0 = time.perf_counter()
    print(f"{objetivo} vs {detectada} ({'acerto' if acerto else 'fallo'}): "
          f"{reaccion_emocion(objetivo, detectada, acerto)}  "
          f"[{time.perf_counter() - t0:.1f}s]")

## 9. Latencia y RAM

El criterio del proyecto es el del README: `qwen2.5:0.5b` se eligio por **484 MB
de RAM y 1-3 s por respuesta** frente a 2.5 GB y 7-12 s de `llama3.2:3b`. Esta
celda repite un mismo mensaje para ver la dispersion, no solo el promedio: en el
turno del juego de emociones la voz ya se lleva 4.8 s de 16.8 s, asi que una cola
larga del LLM rompe la conversacion aunque la mediana este bien.

In [ ]:
import statistics

REPETICIONES = 5
MENSAJE_BENCH = "que te gusta hacer los fines de semana?"

tiempos = []
for i in range(REPETICIONES):
    texto, dt = responder(MENSAJE_BENCH, usar_memoria=False)
    tiempos.append(dt)
    print(f"{i + 1}. [{dt:4.1f}s] {texto}")

print(f"\nmediana {statistics.median(tiempos):.2f}s   "
      f"min {min(tiempos):.2f}s   max {max(tiempos):.2f}s")
if len(tiempos) > 1:
    print(f"desvio  {statistics.stdev(tiempos):.2f}s")
print("referencia README: 1-3 s por respuesta")

In [ ]:
# RAM de los procesos de Ollama. En la Pi es el numero que decide si el modelo
# entra; en esta maquina es solo referencia.
try:
    import psutil

    total = 0
    for proc in psutil.process_iter(["name", "memory_info"]):
        if "ollama" in (proc.info["name"] or "").lower():
            mb = proc.info["memory_info"].rss / 1024 ** 2
            total += mb
            print(f"{proc.info['name']:<24} {mb:7.0f} MB")
    print(f"{'total':<24} {total:7.0f} MB   (referencia README: ~484 MB por modelo)")
except ImportError:
    print("pip install psutil")

## 10. Revisar y etiquetar el registro

`registro_chat.py` guarda cada turno en JSONL con `veredicto` y `correccion` en
`None`, para revisarlos despues. Aviso del README: **nunca se entrena directo con
este archivo**, para que el modelo no refuerce sus propios errores -- las
correcciones se revisan a mano antes de que entren a un dataset.

Lo de abajo lee el registro del notebook (`notebook.jsonl`). Para mirar el del
robot, cambiar la ruta a `~/.lora/chat_libre_training/conversaciones.jsonl`.

In [ ]:
import json

RUTA_REVISAR = Path(registro_chat.REGISTRO)
# RUTA_REVISAR = Path.home() / ".lora" / "chat_libre_training" / "conversaciones.jsonl"

turnos = []
if RUTA_REVISAR.exists():
    for linea in RUTA_REVISAR.read_text(encoding="utf-8").splitlines():
        if linea.strip():
            try:
                turnos.append(json.loads(linea))
            except ValueError:
                pass

print(f"{RUTA_REVISAR}: {len(turnos)} turnos\n")
for t in turnos[-10:]:
    print(f"[{t['ts']}] {t['modelo']}")
    print(f"  > {t['usuario']}")
    print(f"  < {t['respuesta']}")
    if t.get("veredicto") is not None:
        print(f"  veredicto={t['veredicto']} correccion={t.get('correccion')}")

In [ ]:
# Etiquetar un turno a mano: indice (negativo = desde el final), veredicto y, si
# corresponde, como deberia haber contestado.
INDICE = -1
VEREDICTO = "mal"        # "bien" | "mal"
CORRECCION = "Los domingos juego al futbol con amigos."

if turnos:
    turnos[INDICE]["veredicto"] = VEREDICTO
    turnos[INDICE]["correccion"] = CORRECCION or None
    RUTA_REVISAR.write_text(
        "".join(json.dumps(t, ensure_ascii=False) + "\n" for t in turnos),
        encoding="utf-8")
    print(json.dumps(turnos[INDICE], ensure_ascii=False, indent=2))
else:
    print("registro vacio")

## 11. Limpieza

Borra el registro del notebook. No toca el del robot.

In [ ]:
if REGISTRO_NB.exists():
    REGISTRO_NB.unlink()
    print(f"borrado {REGISTRO_NB}")
else:
    print("nada que borrar")
memoria_episodica._cache["mtime"] = None